In [6]:
#in_dir = "/Users/yerik/Music/_4_MUSIC_PROD/_0_PROD_MATERIAL/_ALL_3_PROD_stems_S/_25_07_DRUM/d_40_percent_silence"
in_dir  ="/Users/yerik/Music/_4_MUSIC_PROD/_0_PROD_MATERIAL/_ALL_3_PROD_stems_S"
out_dir = "/Users/yerik/Music/_0_YODJ_PROD/_MIDLIB_3_CLAPS"

# GET ONE CLAPS SAMPLE 

In [7]:
# =========================================================
# -----######-----######  CORE IMPORTABLE FUNCTION  ######-----######-----######
# =========================================================

import os
import re
import shutil
import subprocess
import tempfile

import numpy as np
import pandas as pd
import librosa
import soundfile as sf
from tqdm import tqdm


def _clap_0403_i1_GET_df_claponly_mp3_from_folder(
    in_dir,
    out_dir,
    audio_extensions=(".mp3", ".MP3"),
    # audio
    sr_target=44100,
    hop_length=256,
    n_fft=2048,
    # filename BPM parse (used only as a safety cap)
    bpm_cap_beats=0.70,          # claps can be short but sometimes roomy; tighten to 0.5 if needed
    # start detection
    pre_ms=6,
    min_gap_ms=110,
    # clap specificity (band + shape)
    clap_band=(1500, 10000),     # broad noisy "hand" energy + crack
    low_reject_band=(30, 180),   # reject kick
    hat_reject_band=(11000, 18000), # reject pure hats/air
    clap_over_low_ratio=4.5,     # must beat low
    clap_over_hat_ratio=1.15,    # must have body, not just air
    rms_gate_db=-48,
    peak_gate_db=-26,
    # claps often have a "wider" noisy body (not as tonal)
    flatness_gate=0.20,          # higher = noisier; too high may reject some tight claps
    centroid_min_hz=2200,        # avoid dull thuds
    centroid_max_hz=11000,       # avoid hats
    # tail trimming (variable length)
    min_clap_ms=70,
    max_clap_ms=520,
    end_hold_ms=20,
    end_db_drop=20,
    end_floor_db=-60,
    fade_ms=7,
    # mp3 export
    mp3_bitrate="320k",
    ffmpeg_path="ffmpeg",
    overwrite=True,
    max_files=None,
):
    """
    Recursively find all MP3 files.
    For each MP3:
      - Parse BPM from filename: last '-<BPM>.mp3' (used only for max duration cap)
      - Detect ONE best clap-like transient:
          * dominant 1.5k–10k noisy energy
          * reject low-end dominance (kick)
          * reject ultra-high dominance (hat-only)
          * spectral centroid in clap-ish range
          * spectral flatness gate (noisy burst)
      - Trim tail using clap-band decay => "just the clap", variable length
      - Export as MP3 with SAME filename into out_dir
      - Originals untouched

    Returns:
      df_out: one row per mp3 with timings + export path + debug stats
    """

    os.makedirs(out_dir, exist_ok=True)

    if shutil.which(ffmpeg_path) is None:
        raise RuntimeError(
            f"ffmpeg not found on PATH as '{ffmpeg_path}'. "
            "Install via: brew install ffmpeg  (or pass ffmpeg_path)"
        )

    # gather files recursively
    exts = tuple(audio_extensions) if isinstance(audio_extensions, (list, tuple)) else (audio_extensions,)
    mp3_paths = []
    for root, _, files in os.walk(in_dir):
        for fn in files:
            if fn.startswith("._") or fn.startswith(".DS"):
                continue
            if fn.endswith(exts):
                mp3_paths.append(os.path.join(root, fn))

    mp3_paths = sorted(mp3_paths)
    if max_files is not None:
        mp3_paths = mp3_paths[: int(max_files)]

    # bpm parser: last "-<bpm>.mp3"
    re_bpm = re.compile(r"-([0-9]+(?:\.[0-9]+)?)\.mp3$", re.IGNORECASE)

    def _parse_bpm_from_name(path):
        base = os.path.basename(path)
        m = re_bpm.search(base)
        if not m:
            return None
        bpm = float(m.group(1))
        if bpm <= 0:
            return None
        return bpm

    def _peak_db(y_seg):
        pk = float(np.max(np.abs(y_seg)) + 1e-12)
        return float(20 * np.log10(pk))

    def _band_rms_db(S_mag, freqs, fr0, fr1, f0, f1):
        if fr1 <= fr0:
            return -120.0
        fmask = (freqs >= f0) & (freqs <= f1)
        band = S_mag[fmask, fr0:fr1]
        if band.size == 0:
            return -120.0
        val = float(np.sqrt(np.mean(band ** 2)) + 1e-12)
        return float(20 * np.log10(val))

    def _band_env_db(y_seg, sr, f0, f1):
        S = np.abs(librosa.stft(y_seg, n_fft=n_fft, hop_length=hop_length)) + 1e-12
        freqs = librosa.fft_frequencies(sr=sr, n_fft=n_fft)
        fmask = (freqs >= f0) & (freqs <= f1)
        band_energy = np.sqrt(np.mean(S[fmask, :] ** 2, axis=0)) + 1e-12
        band_db = librosa.amplitude_to_db(band_energy, ref=np.max)
        return band_db

    rows = []

    for src_path in tqdm(mp3_paths, desc="TQM | mp3 → find 1 clap → trim → export mp3", leave=True):
        base_fn = os.path.basename(src_path)
        out_path = os.path.join(out_dir, base_fn)

        try:
            bpm = _parse_bpm_from_name(src_path)
            beat_sec = (60.0 / bpm) if bpm else None

            # duration caps
            max_len_sec = max_clap_ms / 1000.0
            if beat_sec:
                max_len_sec = min(max_len_sec, bpm_cap_beats * beat_sec)

            pre_s = pre_ms / 1000.0
            min_len_sec = min_clap_ms / 1000.0

            # load mono
            y, sr = librosa.load(src_path, sr=sr_target, mono=True)
            if y is None or len(y) < int(sr * 0.25):
                raise ValueError("Audio too short or unreadable.")

            # onset detection
            onset_env = librosa.onset.onset_strength(y=y, sr=sr, hop_length=hop_length)
            onset_frames = librosa.onset.onset_detect(
                onset_envelope=onset_env,
                sr=sr,
                hop_length=hop_length,
                backtrack=False,
                pre_max=5, post_max=5, pre_avg=5, post_avg=5,
                delta=0.18, wait=0
            )
            onset_times = librosa.frames_to_time(onset_frames, sr=sr, hop_length=hop_length)
            if len(onset_times) == 0:
                raise ValueError("No onsets detected.")

            # merge close hits
            min_gap_sec = min_gap_ms / 1000.0
            merged = []
            for t in onset_times:
                t = float(t)
                if not merged or (t - merged[-1]) >= min_gap_sec:
                    merged.append(t)

            # global STFT for band scoring
            S = np.abs(librosa.stft(y, n_fft=n_fft, hop_length=hop_length)) + 1e-12
            freqs = librosa.fft_frequencies(sr=sr, n_fft=n_fft)

            # RMS gate
            rms = librosa.feature.rms(y=y, frame_length=n_fft, hop_length=hop_length)[0]
            rms_db = librosa.amplitude_to_db(rms + 1e-12, ref=np.max)

            # spectral centroid + flatness
            cent = librosa.feature.spectral_centroid(y=y, sr=sr, n_fft=n_fft, hop_length=hop_length)[0]
            flat = librosa.feature.spectral_flatness(y=y, n_fft=n_fft, hop_length=hop_length)[0]

            best = None

            for t in merged:
                start_sec = max(0.0, t - pre_s)
                end_sec_tmp = min(len(y) / sr, start_sec + max_len_sec)

                a0 = int(start_sec * sr)
                a1 = int(end_sec_tmp * sr)
                if a1 <= a0 + int(min_len_sec * sr):
                    continue

                # local gates
                fr = int((t * sr) / hop_length)
                fr0g = max(0, fr - 2)
                fr1g = min(len(rms_db), fr + 3)
                local_rms_db = float(np.max(rms_db[fr0g:fr1g]))

                pk_win = min(len(y), a0 + int(0.09 * sr))
                pk_db = _peak_db(y[a0:pk_win])

                if local_rms_db < rms_gate_db:
                    continue
                if pk_db < peak_gate_db:
                    continue

                # centroid + flatness near onset
                c0 = max(0, fr - 2)
                c1 = min(len(cent), fr + 3)
                local_cent_hz = float(np.max(cent[c0:c1]))
                if not (centroid_min_hz <= local_cent_hz <= centroid_max_hz):
                    continue

                f0 = max(0, fr - 2)
                f1 = min(len(flat), fr + 3)
                local_flat = float(np.max(flat[f0:f1]))
                if local_flat < flatness_gate:
                    continue

                # short window for clap signature (wide noisy burst)
                short_end = min(len(y) / sr, start_sec + 0.14)
                a1s = int(short_end * sr)

                fr0b = max(0, int(a0 / hop_length))
                fr1b = min(S.shape[1], int(a1s / hop_length) + 1)

                cl_db = _band_rms_db(S, freqs, fr0b, fr1b, clap_band[0], clap_band[1])
                lo_db = _band_rms_db(S, freqs, fr0b, fr1b, low_reject_band[0], low_reject_band[1])
                ht_db = _band_rms_db(S, freqs, fr0b, fr1b, hat_reject_band[0], hat_reject_band[1])

                cl_lin = 10 ** (cl_db / 20.0)
                lo_lin = 10 ** (lo_db / 20.0)
                ht_lin = 10 ** (ht_db / 20.0)

                ratio_cl_low = float(cl_lin / (lo_lin + 1e-12))
                ratio_cl_hat = float(cl_lin / (ht_lin + 1e-12))

                if ratio_cl_low < clap_over_low_ratio:
                    continue
                if ratio_cl_hat < clap_over_hat_ratio:
                    continue

                # score: wide/noisy body + punch
                score = float((ratio_cl_low * 1.1) + (ratio_cl_hat * 0.7) + (local_flat * 3.0) + (pk_db / 10.0))

                if (best is None) or (score > best["score"]):
                    best = {
                        "onset_sec": t,
                        "start_sec": start_sec,
                        "score": score,
                        "ratio_cl_low": ratio_cl_low,
                        "ratio_cl_hat": ratio_cl_hat,
                        "centroid_hz": local_cent_hz,
                        "flatness": local_flat,
                        "peak_db_first90ms": pk_db,
                    }

            if best is None:
                raise ValueError("No clap-like onset passed filters (try loosening ratios/gates).")

            # ---- trim end via clap-band decay ----
            start_sec = best["start_sec"]
            start_samp = int(start_sec * sr)

            max_end_sec = min(len(y) / sr, start_sec + max_len_sec)
            max_end_samp = int(max_end_sec * sr)

            seg = y[start_samp:max_end_samp].copy()
            if len(seg) < int(min_len_sec * sr):
                raise ValueError("Segment too short after start selection.")

            cl_env_db = _band_env_db(seg, sr, clap_band[0], clap_band[1])  # 0 at peak

            hold_frames = max(1, int((end_hold_ms / 1000.0) * sr / hop_length))
            peak_frame = int(np.argmax(cl_env_db))

            drop_thr = -abs(end_db_drop)
            floor_thr = float(end_floor_db)

            min_end_samp = int(min_len_sec * sr)
            min_end_frame = max(0, int(min_end_samp / hop_length))

            end_frame = None
            for fr in range(max(min_end_frame, peak_frame + 1), len(cl_env_db) - hold_frames):
                window = cl_env_db[fr : fr + hold_frames]
                if np.all(window <= drop_thr) or np.all(window <= floor_thr):
                    end_frame = fr
                    break

            if end_frame is None:
                fallback_sec = min(max_len_sec, 0.26)  # roomy clap tail
                end_samp_local = int(fallback_sec * sr)
            else:
                end_samp_local = int(end_frame * hop_length)

            end_samp_local = max(end_samp_local, int(min_len_sec * sr))
            end_samp_local = min(end_samp_local, len(seg))

            clip = seg[:end_samp_local].copy()

            # fade out
            fade_len = int((fade_ms / 1000.0) * sr)
            if len(clip) > fade_len + 4:
                fade = np.linspace(1.0, 0.0, fade_len)
                clip[-fade_len:] *= fade

            # export mp3 via ffmpeg
            with tempfile.TemporaryDirectory() as td:
                tmp_wav = os.path.join(td, "tmp.wav")
                sf.write(tmp_wav, clip, sr)

                cmd = [
                    ffmpeg_path, "-y" if overwrite else "-n",
                    "-i", tmp_wav,
                    "-vn",
                    "-ar", str(sr),
                    "-ac", "1",
                    "-b:a", mp3_bitrate,
                    out_path
                ]
                p = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
                if p.returncode != 0:
                    raise RuntimeError(f"ffmpeg failed: {p.stderr[-600:]}")

            rows.append({
                "src_path": src_path,
                "src_file": base_fn,
                "bpm": bpm,
                "beat_sec": beat_sec,
                "onset_sec": best["onset_sec"],
                "start_sec": best["start_sec"],
                "dur_ms": float(len(clip) / sr * 1000.0),
                "score": best["score"],
                "ratio_cl_low": best["ratio_cl_low"],
                "ratio_cl_hat": best["ratio_cl_hat"],
                "centroid_hz": best["centroid_hz"],
                "flatness": best["flatness"],
                "out_path": out_path,
                "error": None
            })

        except Exception as e:
            rows.append({
                "src_path": src_path,
                "src_file": base_fn,
                "bpm": None,
                "beat_sec": None,
                "onset_sec": None,
                "start_sec": None,
                "dur_ms": None,
                "score": None,
                "ratio_cl_low": None,
                "ratio_cl_hat": None,
                "centroid_hz": None,
                "flatness": None,
                "out_path": None,
                "error": str(e)
            })

    return pd.DataFrame(rows)

In [8]:

audio_extensions = (".mp3", ".MP3")

df_claps = _clap_0403_i1_GET_df_claponly_mp3_from_folder(
    in_dir=in_dir,
    out_dir=out_dir,
    audio_extensions=audio_extensions,
    sr_target=44100,
    # start here (balanced)
    clap_over_low_ratio=4.5,
    clap_over_hat_ratio=1.15,
    flatness_gate=0.20,
    centroid_min_hz=2200,
    centroid_max_hz=11000,
    # trimming
    end_db_drop=20,
    end_hold_ms=20,
    min_clap_ms=70,
    max_clap_ms=520,
    # mp3
    mp3_bitrate="320k",
    ffmpeg_path="ffmpeg",
    overwrite=True,
)

TQM | mp3 → find 1 clap → trim → export mp3: 100%|████████████████████| 75549/75549 [9:28:30<00:00,  2.21it/s]


# CUT the ones that are not one CLAPS


In [9]:
folder_path = out_dir

df_hats = _hihat_0403_i1_GET_df_onlyONEhihat_mp3(
    folder_path=folder_path,
    dry_run=True,          # run FIRST
    erase_mode="move",
    verbose=True
)

# when you're happy:
df_hats = _hihat_0403_i1_GET_df_onlyONEhihat_mp3(
    folder_path=folder_path,
    dry_run=False,         # actually moves/deletes
    erase_mode="move",
    verbose=True
)

NameError: name '_hihat_0403_i1_GET_df_onlyONEhihat_mp3' is not defined

# organize CLAPS in folders 

In [10]:
# ============================================================
# 0_FNS
# ============================================================

import os
import shutil
import numpy as np
import pandas as pd
from tqdm import tqdm
import librosa


# -----######-----######  CORE IMPORTABLE FUNCTION  #####-----######-----######
def _hat_0403_smartbucket_inplace_GET_df_manifest(
    in_dir,
    audio_extensions,
    mode="move",                  # "move" | "copy" | "none"
    sr_target=44100,
    top_db_trim=70,               # hats often have quiet tails; don't trim too aggressively
    n_fft=4096,                   # better HF resolution
    hop_length=256,
    min_per_bucket=10,            # prevents empty folders
    dry_run=False,
    seed=7,
):
    """
    Smart hi-hat organizer (in-place):
    - Reads audio files from in_dir (root only)
    - Creates hat bucket folders inside in_dir
    - Assigns with hat-specific multi-check scoring (HF bands are key)
    - Rebalances borderline files to avoid empty folders
    - Moves (or copies) originals into bucket folders
    - Saves hat_manifest.csv into in_dir
    Returns: df_manifest, df_summary
    """

    rng = np.random.RandomState(seed)

    # --- Hat folders (production-first) ---
    # (You can rename these later, but these buckets are actually useful in practice.)
    buckets = [
        "01_CLOSED_TIGHT",         # short, clean tick
        "02_CLOSED_TICKY",         # clicky / bright / crisp
        "03_OPEN_WASHY",           # long decay, airy
        "04_OPEN_METALLIC",        # long + tonal peaks / metallic ring
        "05_SHAKER_HAT",           # shaker-like noise band
        "06_NOISY_TEXTURE",        # noisy / flat / gritty
        "07_CRASHY_HAT",           # very long, wideband (almost cymbal-ish)
        "08_LOFI_DIRTY",           # dirt + rolled highs / crunchy
        "09_WEIRD_FX",             # leftover catcher for oddities
    ]
    bucket_paths = {b: os.path.join(in_dir, b) for b in buckets}

    # ---------------- helpers ----------------
    def _safe_makedirs(p):
        os.makedirs(p, exist_ok=True)

    def _is_in_bucket_folder(path_abs):
        for b in buckets:
            b_abs = os.path.abspath(bucket_paths[b])
            if os.path.abspath(path_abs).startswith(b_abs + os.sep):
                return True
        return False

    def _list_audio_files(root, exts):
        exts_l = [e.lower() for e in exts] if exts else []
        paths = []
        for fn in os.listdir(root):
            if fn.startswith("._") or fn.startswith(".DS"):
                continue
            p = os.path.join(root, fn)
            if os.path.isfile(p):
                if _is_in_bucket_folder(p):
                    continue
                ext = os.path.splitext(fn)[1].lower()
                if (not exts_l) or (ext in exts_l):
                    paths.append(p)
        return sorted(paths)

    def _mk_dest(dst_dir, src):
        dst = os.path.join(dst_dir, os.path.basename(src))
        if not os.path.exists(dst):
            return dst
        base, ext = os.path.splitext(os.path.basename(src))
        i = 1
        while True:
            dst2 = os.path.join(dst_dir, f"{base}__DUP{i}{ext}")
            if not os.path.exists(dst2):
                return dst2
            i += 1

    def _band_energy_ratio(S, freqs, f_lo, f_hi):
        mask = (freqs >= f_lo) & (freqs < f_hi)
        if not np.any(mask):
            return 0.0
        num = float(np.sum(S[mask, :]))
        den = float(np.sum(S)) + 1e-12
        return num / den

    def _decay_ms_from_peak(x, sr, drop_db):
        env = np.abs(x)
        if env.size < 10:
            return 0.0
        win = max(16, int(0.004 * sr))  # ~4ms smoothing
        k = np.ones(win) / win
        env_s = np.convolve(env, k, mode="same")

        peak_idx = int(np.argmax(env_s))
        peak_val = float(env_s[peak_idx]) + 1e-12
        target = peak_val * (10 ** (-drop_db / 20.0))

        tail = env_s[peak_idx:]
        below = np.where(tail <= target)[0]
        if below.size == 0:
            return (len(tail) / sr) * 1000.0
        return (float(below[0]) / sr) * 1000.0

    def _q(series, p):
        return float(np.nanpercentile(series.to_numpy(dtype=float), p))

    def _z_factory(df_ok, col):
        v = df_ok[col].to_numpy(dtype=float)
        mu = float(np.nanmean(v))
        sd = float(np.nanstd(v) + 1e-12)
        def _z(vv):
            return (float(vv) - mu) / sd if sd > 1e-12 else 0.0
        return _z

    # ---------------- setup folders ----------------
    for b in buckets:
        _safe_makedirs(bucket_paths[b])

    paths = _list_audio_files(in_dir, audio_extensions)

    # ---------------- feature extraction ----------------
    rows = []
    for p in tqdm(paths, desc="Extracting hi-hat features", total=len(paths)):
        try:
            y, sr = librosa.load(p, sr=sr_target, mono=True)
            y, _ = librosa.effects.trim(y, top_db=top_db_trim)
            if y.size == 0:
                raise ValueError("empty_audio_after_trim")

            y = y / (np.max(np.abs(y)) + 1e-12)

            # STFT magnitude
            S = np.abs(librosa.stft(y, n_fft=n_fft, hop_length=hop_length)) + 1e-12
            freqs = librosa.fft_frequencies(sr=sr, n_fft=n_fft)

            # ---- THE IMPORTANT BANDS FOR HATS ----
            low_ratio   = _band_energy_ratio(S, freqs, 20, 300)         # hats should be low
            mid_ratio   = _band_energy_ratio(S, freqs, 300, 2000)
            pres_ratio  = _band_energy_ratio(S, freqs, 2000, 6000)      # presence
            hf_ratio    = _band_energy_ratio(S, freqs, 6000, 16000)     # hi-hat "air/metal"
            air_ratio   = _band_energy_ratio(S, freqs, 10000, 20000)    # air band (if sr supports)

            centroid = float(np.mean(librosa.feature.spectral_centroid(S=S, sr=sr)))
            rolloff  = float(np.mean(librosa.feature.spectral_rolloff(S=S, sr=sr, roll_percent=0.90)))
            flatness = float(np.mean(librosa.feature.spectral_flatness(S=S)))
            zcr      = float(np.mean(librosa.feature.zero_crossing_rate(y)))

            # decay + duration
            decay_ms_24 = _decay_ms_from_peak(y, sr, drop_db=24)        # hats: longer tails matter
            decay_ms_12 = _decay_ms_from_peak(y, sr, drop_db=12)        # early decay
            dur_ms      = (len(y) / sr) * 1000.0

            # transient sharpness proxy (first 30ms vs whole)
            early = y[: min(len(y), int(0.03 * sr))]
            early_rms = float(np.sqrt(np.mean(early**2) + 1e-12))
            full_rms  = float(np.sqrt(np.mean(y**2) + 1e-12))
            sharp = float(early_rms / (full_rms + 1e-12))

            # tonal/metallic indicator:
            # metallic hats often show clearer peaks => lower flatness + higher centroid/rolloff
            metallic_hint = float((1.0 - flatness) * (centroid / (rolloff + 1e-9)))

            rows.append({
                "Path": p,
                "file_name": os.path.basename(p),
                "dur_ms": dur_ms,
                "decay12_ms": decay_ms_12,
                "decay24_ms": decay_ms_24,
                "low_ratio": low_ratio,
                "mid_ratio": mid_ratio,
                "pres_ratio": pres_ratio,
                "hf_ratio": hf_ratio,
                "air_ratio": air_ratio,
                "centroid_hz": centroid,
                "rolloff_hz": rolloff,
                "flatness": flatness,
                "zcr": zcr,
                "sharp": sharp,
                "metallic_hint": metallic_hint,
                "error": ""
            })

        except Exception as e:
            rows.append({
                "Path": p,
                "file_name": os.path.basename(p),
                "dur_ms": np.nan,
                "decay12_ms": np.nan,
                "decay24_ms": np.nan,
                "low_ratio": np.nan,
                "mid_ratio": np.nan,
                "pres_ratio": np.nan,
                "hf_ratio": np.nan,
                "air_ratio": np.nan,
                "centroid_hz": np.nan,
                "rolloff_hz": np.nan,
                "flatness": np.nan,
                "zcr": np.nan,
                "sharp": np.nan,
                "metallic_hint": np.nan,
                "error": str(e)
            })

    df = pd.DataFrame(rows)
    df_ok = df[df["error"].eq("")].copy()

    if len(df_ok) == 0:
        df.to_csv(os.path.join(in_dir, "hat_manifest.csv"), index=False)
        summary = pd.DataFrame({"bucket": buckets, "count": [0]*len(buckets)})
        return df, summary

    # ---------------- adaptive percentiles ----------------
    Q = {
        "dur_lo":    _q(df_ok["dur_ms"], 25),
        "dur_hi":    _q(df_ok["dur_ms"], 75),
        "d24_lo":    _q(df_ok["decay24_ms"], 25),
        "d24_hi":    _q(df_ok["decay24_ms"], 75),
        "hf_hi":     _q(df_ok["hf_ratio"], 75),
        "hf_lo":     _q(df_ok["hf_ratio"], 25),
        "air_hi":    _q(df_ok["air_ratio"], 75),
        "flat_hi":   _q(df_ok["flatness"], 75),
        "flat_lo":   _q(df_ok["flatness"], 25),
        "low_hi":    _q(df_ok["low_ratio"], 75),
        "sharp_hi":  _q(df_ok["sharp"], 75),
        "cent_hi":   _q(df_ok["centroid_hz"], 75),
        "roll_hi":   _q(df_ok["rolloff_hz"], 75),
    }

    # z-score factories
    z_dur   = _z_factory(df_ok, "dur_ms")
    z_d24   = _z_factory(df_ok, "decay24_ms")
    z_hf    = _z_factory(df_ok, "hf_ratio")
    z_air   = _z_factory(df_ok, "air_ratio")
    z_low   = _z_factory(df_ok, "low_ratio")
    z_flat  = _z_factory(df_ok, "flatness")
    z_zcr   = _z_factory(df_ok, "zcr")
    z_sharp = _z_factory(df_ok, "sharp")
    z_cent  = _z_factory(df_ok, "centroid_hz")
    z_roll  = _z_factory(df_ok, "rolloff_hz")
    z_metl  = _z_factory(df_ok, "metallic_hint")

    # ---------------- scoring model (hat-specific) ----------------
    # We don’t do "if-else". We compute scores for every bucket and pick the max.
    def _scores(r):
        dur   = float(r["dur_ms"])
        d24   = float(r["decay24_ms"])
        hf    = float(r["hf_ratio"])
        air   = float(r["air_ratio"])
        low   = float(r["low_ratio"])
        flat  = float(r["flatness"])
        zcr   = float(r["zcr"])
        sharp = float(r["sharp"])
        cent  = float(r["centroid_hz"])
        roll  = float(r["rolloff_hz"])
        metl  = float(r["metallic_hint"])

        # gates (soft bonuses)
        gate_hf   = 0.5 if hf >= Q["hf_hi"] else 0.0
        gate_air  = 0.4 if air >= Q["air_hi"] else 0.0
        gate_tight= 0.6 if (d24 <= Q["d24_lo"] and dur <= Q["dur_lo"]) else 0.0
        gate_open = 0.6 if (d24 >= Q["d24_hi"] and dur >= Q["dur_hi"]) else 0.0
        gate_noisy= 0.6 if (flat >= Q["flat_hi"] and zcr >= _q(df_ok["zcr"], 75)) else 0.0

        sc = {}

        # 01 Closed Tight: short tail, sharp transient, HF present but controlled
        sc["01_CLOSED_TIGHT"] = (
            -1.6 * z_d24(d24) -
            1.0 * z_dur(dur) +
            1.2 * z_sharp(sharp) +
            0.6 * z_hf(hf) -
            0.5 * z_low(low)
        ) + gate_tight

        # 02 Closed Ticky: very crisp HF/air, sharp, still short-ish
        sc["02_CLOSED_TICKY"] = (
            1.4 * z_hf(hf) +
            1.0 * z_air(air) +
            0.9 * z_sharp(sharp) -
            1.0 * z_d24(d24) -
            0.4 * z_low(low)
        ) + gate_hf + gate_air

        # 03 Open Washy: long tail, lots of HF air, more noise-like than metallic
        sc["03_OPEN_WASHY"] = (
            1.6 * z_d24(d24) +
            1.1 * z_dur(dur) +
            0.9 * z_hf(hf) +
            0.6 * z_air(air) +
            0.5 * z_flat(flat)
        ) + gate_open

        # 04 Open Metallic: long-ish, more “peaky/metallic” (lower flatness, higher rolloff/centroid)
        sc["04_OPEN_METALLIC"] = (
            1.3 * z_d24(d24) +
            0.8 * z_dur(dur) +
            1.0 * z_cent(cent) +
            1.0 * z_roll(roll) -
            0.9 * z_flat(flat) +
            0.8 * z_metl(metl)
        ) + gate_open

        # 05 Shaker Hat: noise band focused, mid presence, high zcr, not too metallic
        sc["05_SHAKER_HAT"] = (
            1.2 * z_flat(flat) +
            1.0 * z_zcr(zcr) +
            0.6 * z_hf(hf) -
            0.6 * z_metl(metl) -
            0.4 * z_d24(d24)
        ) + (0.4 if (flat >= Q["flat_lo"] and hf >= Q["hf_lo"]) else 0.0)

        # 06 Noisy Texture: very flat/noisy, strong zcr, wideband
        sc["06_NOISY_TEXTURE"] = (
            1.7 * z_flat(flat) +
            1.3 * z_zcr(zcr) +
            0.5 * z_roll(roll) +
            0.4 * z_hf(hf)
        ) + gate_noisy

        # 07 Crashy Hat: very long + very wideband HF/air (almost cymbal)
        sc["07_CRASHY_HAT"] = (
            1.8 * z_d24(d24) +
            1.4 * z_dur(dur) +
            1.0 * z_roll(roll) +
            0.9 * z_air(air) +
            0.6 * z_hf(hf)
        ) + gate_open + gate_air

        # 08 Lofi Dirty: less HF/air, more low/mid junk, sometimes noisy
        sc["08_LOFI_DIRTY"] = (
            1.2 * z_low(low) -
            1.2 * z_hf(hf) -
            0.8 * z_air(air) +
            0.8 * z_flat(flat)
        ) + (0.4 if (low >= Q["low_hi"] and hf <= Q["hf_lo"]) else 0.0)

        # 09 Weird FX: catcher for unusual combos (high metallic + low hf, etc.)
        weird_bonus = 0.0
        if (metl >= _q(df_ok["metallic_hint"], 75) and hf <= Q["hf_lo"]):
            weird_bonus += 0.6
        if (flat <= Q["flat_lo"] and d24 >= Q["d24_hi"] and air <= _q(df_ok["air_ratio"], 25)):
            weird_bonus += 0.4
        sc["09_WEIRD_FX"] = (
            0.4 * z_metl(metl) +
            0.3 * z_cent(cent) +
            0.2 * z_flat(flat) +
            0.2 * z_d24(d24)
        ) + weird_bonus

        return sc

    # assign bucket + confidence margin
    assigned = []
    conf = []
    score_rows = []

    for r in df_ok.to_dict("records"):
        sc = _scores(r)
        items = sorted(sc.items(), key=lambda kv: kv[1], reverse=True)
        b1, s1 = items[0]
        b2, s2 = items[1]
        assigned.append(b1)
        conf.append(float(s1 - s2))
        score_rows.append(sc)

    df_ok["bucket"] = assigned
    df_ok["conf_margin"] = conf
    df_ok["_idx"] = np.arange(len(df_ok))

    score_df = pd.DataFrame(score_rows)
    score_df["_idx"] = df_ok["_idx"].values

    # ---------------- rebalance: avoid empty folders (only move borderline samples) ----------------
    def _rebalance_once(df_work):
        counts = df_work["bucket"].value_counts().to_dict()
        need = [b for b in buckets if counts.get(b, 0) < min_per_bucket]
        if not need:
            return df_work, False

        df_cand = df_work.sort_values("conf_margin", ascending=True).copy()
        moved = False

        for target in need:
            cur = counts.get(target, 0)
            add = max(0, min_per_bucket - cur)
            if add == 0:
                continue

            merged = df_cand.merge(score_df[["_idx", target]], on="_idx", how="left")
            merged = merged.rename(columns={target: "target_score"})
            merged = merged[merged["bucket"] != target].copy()

            merged = merged.sort_values(["target_score", "conf_margin"], ascending=[False, True])
            pick = merged.head(add)
            if len(pick) == 0:
                continue

            idxs = pick["_idx"].to_list()
            df_work.loc[df_work["_idx"].isin(idxs), "bucket"] = target
            moved = True

        return df_work, moved

    for _ in range(6):
        df_ok, changed = _rebalance_once(df_ok)
        if not changed:
            break

    # merge back
    df = df.merge(df_ok[["Path", "bucket", "conf_margin"]], on="Path", how="left")
    df["bucket"] = df["bucket"].fillna("09_WEIRD_FX")

    # ---------------- move/copy in-place ----------------
    if (mode.lower() in ["move", "copy"]) and (not dry_run):
        ok_rows = df[df["error"].eq("")]
        for r in tqdm(ok_rows.itertuples(index=False), desc=f"{mode.upper()} hats into folders", total=len(ok_rows)):
            src = r.Path
            bucket = r.bucket if isinstance(r.bucket, str) else "09_WEIRD_FX"
            dst_dir = bucket_paths.get(bucket, bucket_paths["09_WEIRD_FX"])
            dst = _mk_dest(dst_dir, src)

            if os.path.abspath(src) == os.path.abspath(dst):
                continue

            if mode.lower() == "copy":
                shutil.copy2(src, dst)
            else:
                shutil.move(src, dst)

    # ---------------- save manifest + summary ----------------
    out_csv = os.path.join(in_dir, "hat_manifest.csv")
    df.to_csv(out_csv, index=False)

    counts = df[df["error"].eq("")]["bucket"].value_counts().reindex(buckets).fillna(0).astype(int)
    df_summary = counts.reset_index()
    df_summary.columns = ["bucket", "count"]

    return df, df_summary

In [11]:
# ============================================================
#!#!#!#!#! RUNNING STATEMENTS #!#!#!#!#!
# ============================================================

audio_extensions = [".mp3"]  # add more if you want

df_hats, df_summary = _hat_0403_smartbucket_inplace_GET_df_manifest(
    in_dir=out_dir,
    audio_extensions=audio_extensions,
    mode="move",          # moves originals into created subfolders
    min_per_bucket=10,    # raise/lower depending on folder size
    dry_run=False
)

print(df_summary)
print(df_hats[["file_name","bucket","conf_margin","hf_ratio","air_ratio","low_ratio","flatness","decay24_ms","sharp","error"]].head(30))

MOVE hats into folders: 100%|██████████████████████████████████████████████████| 1/1 [00:00<00:00, 248.63it/s]

             bucket  count
0   01_CLOSED_TIGHT      0
1   02_CLOSED_TICKY      0
2     03_OPEN_WASHY      0
3  04_OPEN_METALLIC      0
4     05_SHAKER_HAT      0
5  06_NOISY_TEXTURE      0
6     07_CRASHY_HAT      0
7     08_LOFI_DIRTY      1
8       09_WEIRD_FX      0
                                           file_name         bucket  \
0  _s21_d5-7141w-9BGmaj---25-VARIOUS-0218srcdrums...  08_LOFI_DIRTY   

   conf_margin  hf_ratio  air_ratio  low_ratio  flatness  decay24_ms  \
0          0.0  0.548769   0.435877   0.005058  0.105946  245.079365   

      sharp error  
0  1.265274        


In [12]:
# END 
print("HATS - DONE")

HATS - DONE
